# 🔍 G-LRAG Retrieval — Advanced Edition (Google Colab)

Run the **G-LRAG hybrid retrieval** pipeline with an **advanced retrieval layer**
on Google Colab.

> **What's new vs [`retrieval_colab.ipynb`](notebooks/retrieval_colab.ipynb):**
> This notebook integrates the advanced approaches your teammate prototyped in
> [`hybridrag-advanced.ipynb`](notebooks/hybridrag-advanced.ipynb) — **on top of**
> the repo's production [`HybridRetriever`](src/retrieval/retriever.py) — without
> re-implementing the legs/reranker/graph expander that already exist in the codebase.

### Advanced techniques layered in

| Technique | Source | What it does |
|---|---|---|
| **Multi-query variants** | teammate | For one question, generate *several* queries (original, keyword rewrite, legal expansion, clause splits, optional HyDE) and fuse their rankings — boosts recall for multi-clause / verbose legal questions. |
| **Vietnamese legal query expansion** | teammate | Domain synonyms (`hỗ trợ`→`chính sách hỗ trợ doanh nghiệp nhỏ và vừa`, `thuế`→`thuế thu nhập doanh nghiệp thuế giá trị gia tăng`, …) injected into the keyword variant. |
| **Weighted RRF** | teammate | Generalises the repo's unweighted [`rrf_fuse`](src/retrieval/rrf.py) with per-variant weights (original 1.00, keyword 0.92, expansion 0.82, clause 0.86, HyDE 0.62). |
| **Article aggregation** | teammate | After rerank, group chunks by `law_id|ten_van_ban|dieu_so`, add support-count + RRF + early-rank bonuses, and pick the best chunk per article. Adaptive top-k widens for long / multi-clause questions. |
| **HyDE (optional)** | teammate | Generate a hypothetical legal passage with Qwen2.5-7B and use it as an extra dense-only query variant. Off by default (needs a 2nd GPU model). |

### Why this layer sits *on top of* `HybridRetriever`

The repo already implements lexical (FTS5) + dense (FAISS) + graph expand + cross-encoder
rerank as reusable methods ([`_lexical_leg`](src/retrieval/retriever.py), [`_dense_leg`](src/retrieval/retriever.py),
[`_rerank`](src/retrieval/retriever.py), [`_fetch_meta`](src/retrieval/retriever.py)). The advanced layer simply
**calls those methods once per query variant** and fuses the results with weighted RRF — so the
production code stays the single source of truth for each leg, and this notebook is pure *orchestration*.

The output is still a `List[Hit]`, so the existing [`make_relevant_lists`](src/retrieval/retriever.py)
and the F2 evaluation cell work unchanged.


## 📦 Input data you must provide (on Google Drive)

Put these files in a Google Drive folder, e.g.
`/content/drive/MyDrive/Road2AI_ApplePie/data/stage6_data/`.
Set `DATA_DIR` in the Setup cell to that folder.

| File | Size | Required? | Purpose |
|---|---|---|---|
| `chunk_store.sqlite` | ~2.7 GB | ✅ **Always** | Lexical BM25/FTS5 index + full `chunk_text` + metadata |
| `faiss_index__BAAI_bge-m3.index` | ~2.4 GB | ⛔ Only for dense leg | FAISS `IndexFlatIP` (636,585 × 1024, L2-normalised) |
| `chunk_meta_slim.parquet` | ~86 MB | ⛔ Only for dense leg | Metadata sidecar aligning FAISS position → `row_idx` |
| `embed_model_meta__BAAI_bge-m3.json` | 156 B | ⛔ Only for dense leg | Bundle-compat check (model name, dim, count) |
| `kg.gpickle` | ~tens of MB | ⛔ Only for graph expand | `networkx.MultiDiGraph` knowledge graph |
| `dev_set/questions.json` | ~3.5 KB | optional | 20 example dev-set questions |
| `dev_set/ground_truth.json` | ~8.3 KB | optional | Gold answers for the optional F2 evaluation cell |

> **Minimum:** just `chunk_store.sqlite` for a lexical-only run.
> **Full hybrid + advanced:** all four Stage-6 files (~5.2 GB) + optionally `kg.gpickle`.
> **+ HyDE:** also enables a second GPU model (Qwen2.5-7B) — see `USE_HYDE` below.

> 💡 These artifacts are produced by the local Stage-6 pipeline
> (`Road2AI_ApplePie/data/stage6_data/`) and are git-ignored (too big).
> Upload them to Drive once and reuse across Colab sessions.


## ⚙️ What the notebook needs from Colab

- **Runtime type:** `T4 GPU` (or any GPU runtime) — **required**. The dense leg runs
  `BAAI/bge-m3` and the reranker runs `BAAI/bge-reranker-v2-m3`, both on CUDA.
  If you enable HyDE (`USE_HYDE = True`) a second model (`Qwen/Qwen2.5-7B-Instruct`)
  is also loaded on GPU — a T4 (16 GB) fits reranker + HyDE together, but watch
  memory. Set via *Runtime → Change runtime type → T4 GPU* before running.
- **Disk:** the artifacts total ~5.2 GB; the default ~100 GB Colab disk is fine.
- **RAM:** standard Colab (~12 GB) works. The FAISS `IndexFlatIP` is moved onto
  the T4 GPU to free host RAM.


## 1. Setup — clone repo, mount Drive, install deps, configure

> ⚠️ **Edit the `Configuration` block** — especially `DATA_DIR`, and `USE_HYDE`
> if you want the hypothetical-document query variant.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ===== Configuration (edit these) =====================================
GITHUB_REPO = "https://github.com/vkb0205/Road2AI_ApplePie.git"
REPO_BRANCH = "main"
REPO_DIR    = "/content/Road2AI_ApplePie"

# Folder on your Google Drive that holds the Stage-6 artifacts + dev_set.
DATA_DIR = "/content/drive/Shareddrives/R2AI/data/stage6_data"
DEV_DIR  = "/content/Road2AI_ApplePie/dev_set"

# --- Pipeline switches (legs) -----------------------------------------
USE_DENSE  = True    # load FAISS + BGE-m3 query encoder (runs on GPU + ~2.4GB)
USE_GRAPH  = True   # True -> load kg.gpickle for graph expansion
USE_RERANK = True    # cross-encoder rerank (BAAI/bge-reranker-v2-m3, on GPU)
FTS_MODE   = "bm25_ranked"   # 'bm25_ranked' (baseline) or 'fts_fast' (faster, weaker)

# --- Advanced layer switches (the integrated techniques) ---------------
USE_MULTIQUERY  = True   # multi-query variants + Vietnamese legal expansion
USE_HYDE        = True  # HyDE hypothetical-document variant (needs Qwen2.5-7B)
USE_ARTICLE_AGG = True   # article aggregation with adaptive top-k after rerank

# Advanced retrieval knobs (mirrors hybridrag-advanced.ipynb CFG; safe defaults)
RRF_K              = 60     # RRF smoothing constant
BM25_TOPK          = 40     # lexical candidates per variant
DENSE_TOPK         = 100    # dense candidates per variant
CANDIDATE_TOPK     = 96     # fused candidates kept before rerank
RERANK_TOPK        = 48     # kept after cross-encoder rerank
ARTICLE_CTX_TOPK   = 16     # base articles kept after aggregation
ARTICLE_CTX_MAX    = 24     # cap after adaptive widening
FINAL_TOP_K        = 10     # final hits returned
MAX_QUERY_VARIANTS = 8      # cap on generated variants per question
HYDE_MAX_NEW_TOKENS = 192   # only used if USE_HYDE

# --- HyDE model (only loaded if USE_HYDE) ------------------------------
HYDE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# --- Model download speedups (opt-in, big time saver) ------------------
FAST_DOWNLOAD     = True
HF_CACHE_ON_DRIVE = True
HF_TOKEN          = ""   # or set the Colab secret `HF_TOKEN` and leave this ''
HF_CACHE_DIR = "/content/drive/Shareddrives/R2AI/data/cache_advanced"  # if HF_CACHE_ON_DRIVE
# ======================================================================


In [4]:
# --- 0. GPU runtime check (fail fast) ---------------------------------
import subprocess, sys
gpu_ok = subprocess.run("nvidia-smi", shell=True,
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not gpu_ok:
    raise SystemExit(
        "❌ No GPU detected. This notebook is configured for a GPU runtime.\n"
        "   In Colab: Runtime → Change runtime type → T4 GPU, then restart and run all."
    )
!nvidia-smi -L
print("[gpu] GPU runtime confirmed.")


GPU 0: Tesla T4 (UUID: GPU-58a72240-2a54-a637-b129-4c2153d0db33)
[gpu] GPU runtime confirmed.


In [5]:
import os, sys, time, shutil
from pathlib import Path

# --- 1a. Clone the retrieval source code (lightweight, ~MB) -----------
if not Path(REPO_DIR).exists():
    print(f"[clone] {GITHUB_REPO} -> {REPO_DIR}")
    !git clone --depth 1 -b {REPO_BRANCH} {GITHUB_REPO} {REPO_DIR}
else:
    print(f"[clone] {REPO_DIR} already present")

SRC = Path(REPO_DIR) / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print("[src on path]", SRC)

# --- 1b. Mount Google Drive for the big artifacts ----------------------
from google.colab import drive
drive.mount('/content/drive')

DATA = Path(DATA_DIR)
DEV  = Path(DEV_DIR)
print("[DATA_DIR exists]", DATA.exists(), DATA)
print("[DEV_DIR  exists]", DEV.exists(),  DEV)


[clone] https://github.com/vkb0205/Road2AI_ApplePie.git -> /content/Road2AI_ApplePie
Cloning into '/content/Road2AI_ApplePie'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 78 (delta 4), reused 53 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 250.47 KiB | 1.94 MiB/s, done.
Resolving deltas: 100% (4/4), done.
[src on path] /content/Road2AI_ApplePie/src
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[DATA_DIR exists] True /content/drive/Shareddrives/R2AI/data/stage6_data
[DEV_DIR  exists] True /content/Road2AI_ApplePie/dev_set


In [6]:
# --- 1c. Install dependencies -----------------------------------------
# Light deps are always needed (pandas/pyarrow/networkx are usually preinstalled
# on Colab; we ensure them). Heavy GPU deps only when the corresponding leg
# or the HyDE variant is enabled.
!pip -q install pandas pyarrow networkx pyyaml python-dotenv psutil

# --- Model download speedups (must run before any model load) -----------
if FAST_DOWNLOAD:
    !pip -q install hf_transfer
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

if HF_CACHE_ON_DRIVE:
    os.makedirs(HF_CACHE_DIR, exist_ok=True)
    os.environ["HF_HOME"] = HF_CACHE_DIR
    os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE_DIR
    os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR

# Optional HF token: try the Colab secret first, else the value set above.
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok: os.environ["HF_TOKEN"] = _tok
except Exception:
    if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN

print("[hf] transfer=" + ("on" if FAST_DOWNLOAD else "off")
      + " | cache=" + (HF_CACHE_DIR if HF_CACHE_ON_DRIVE else "ephemeral")
      + " | token=" + ("yes" if os.environ.get("HF_TOKEN") else "no"))

if USE_DENSE or USE_RERANK:
    print("[install] dense/rerank GPU deps (faiss-gpu, FlagEmbedding, torch+CUDA) ...")
    # Compatible pins for FlagEmbedding (bge-m3 / bge-reranker-v2-m3):
    #   * transformers < 4.46 (FlagEmbedding 1.2.x uses prepare_for_model, removed in 4.46)
    #   * FlagEmbedding < 1.3 (1.3 restructured imports and breaks `from FlagEmbedding import ...`)
    #   * transformers >= 4.41 (keeps both prepare_for_model and the `dtype` kwarg)
    !pip -q install "faiss-gpu-cu12>=1.7.2" "FlagEmbedding>=1.2.10,<1.3" "numpy>=1.24" "transformers>=4.41,<4.46"
    !pip -q install torch --index-url https://download.pytorch.org/whl/cu121
    import torch as _t
    print("[torch]", _t.__version__, "| cuda available:", _t.cuda.is_available(),
          "| device:", _t.cuda.get_device_name(0) if _t.cuda.is_available() else "CPU")
    assert _t.cuda.is_available(), (
        "torch installed but CUDA not available. Make sure the Colab runtime "
        "is a GPU runtime (Runtime → Change runtime type → T4 GPU)."
    )

# HyDE pulls in accelerate + bitsandbytes for a comfortable Qwen load.
if USE_HYDE:
    print("[install] HyDE deps (accelerate, sentencepiece, bitsandbytes) ...")
    !pip -q install accelerate sentencepiece bitsandbytes

print("[deps] ready")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 19.9 MB/s eta 0:00:00
[hf] transfer=on | cache=/content/drive/Shareddrives/R2AI/data/cache_advanced | token=no
[install] dense/rerank GPU deps (faiss-gpu, FlagEmbedding, torch+CUDA) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.1/147.1 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 97.0 MB/s eta 0:00:00
[torch] 2.11.0+cu128 | cuda available: True | device: Tesla T4
[install] HyDE deps (accelerate, sentencepiece, bitsandbytes) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00
[deps] ready


## 2. Verify your input data is present

Run this to confirm the files you put on Drive are found before loading.


In [7]:
required = {"chunk_store.sqlite": DATA}
if USE_DENSE:
    required.update({
        "faiss_index__BAAI_bge-m3.index": DATA,
        "chunk_meta_slim.parquet": DATA,
        "embed_model_meta__BAAI_bge-m3.json": DATA,
    })
if USE_GRAPH:
    required["kg.gpickle"] = DATA

ok = True
for name, d in required.items():
    p = d / name
    mb = (p.stat().st_size / 1e6) if p.exists() else 0
    flag = "OK " if p.exists() else "MISSING"
    if not p.exists(): ok = False
    print(f"{flag}  {p}  ({mb:.1f} MB)")

if DEV.exists():
    print(f"OK   {DEV/'questions.json'}")
    print(f"OK   {DEV/'ground_truth.json'}")
else:
    print(f"(optional dev_set not found at {DEV})")

assert ok, "❌ Missing required input files — upload them to DATA_DIR on Drive (see the top table)."


OK   /content/drive/Shareddrives/R2AI/data/stage6_data/chunk_store.sqlite  (2918.4 MB)
OK   /content/drive/Shareddrives/R2AI/data/stage6_data/faiss_index__BAAI_bge-m3.index  (2607.5 MB)
OK   /content/drive/Shareddrives/R2AI/data/stage6_data/chunk_meta_slim.parquet  (89.8 MB)
OK   /content/drive/Shareddrives/R2AI/data/stage6_data/embed_model_meta__BAAI_bge-m3.json  (0.0 MB)
OK   /content/drive/Shareddrives/R2AI/data/stage6_data/kg.gpickle  (431.7 MB)
OK   /content/Road2AI_ApplePie/dev_set/questions.json
OK   /content/Road2AI_ApplePie/dev_set/ground_truth.json


## 3. Build the base retriever (lexical + dense + graph legs)

This builds the repo's [`HybridRetriever`](src/retrieval/retriever.py) exactly as
`retrieval_colab.ipynb` does — the lexical FTS leg, the (optional) dense FAISS
leg (with the index moved onto the T4 GPU), the (optional) graph expander, and
the cross-encoder reranker (loaded lazily on first use).

The advanced layer (Section 4) wraps this object and reuses its legs/reranker.


In [8]:
from retrieval.bm25_index import FTSIndex
from retrieval.faiss_index import FAISSIndex, BGEQueryEncoder
from retrieval.graph_expand import GraphExpander
from retrieval.retriever import HybridRetriever, RetrievalConfig, make_relevant_lists

DB         = DATA / "chunk_store.sqlite"
FAISS_IDX  = DATA / "faiss_index__BAAI_bge-m3.index"
META       = DATA / "chunk_meta_slim.parquet"
MODEL_META = DATA / "embed_model_meta__BAAI_bge-m3.json"
KG         = DATA / "kg.gpickle"

# --- lexical leg (always on) ------------------------------------------
t0 = time.time()
fts = FTSIndex(str(DB), mode=FTS_MODE).open()
print(f"[fts] rows={fts.n_rows:,}  backend={fts.lexical_backend!r}  mode={FTS_MODE!r}  ({time.time()-t0:.1f}s)")

# --- GPU device used by the dense/rerank/HyDE legs ---------------------
if USE_DENSE or USE_RERANK or USE_HYDE:
    import torch
    _dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[gpu] active device = {_dev}" + (f" ({torch.cuda.get_device_name(0)})" if _dev.type=='cuda' else ""))

# --- dense leg (optional, GPU) ----------------------------------------
faiss_index = None
query_encoder = None
if USE_DENSE:
    import psutil
    t0 = time.time()
    faiss_index = FAISSIndex(str(FAISS_IDX), str(META), str(MODEL_META)).load_index()
    # Move the flat index onto GPU 0 and drop the CPU copy -> ~2.6 GB host freed.
    _on_gpu = False
    if torch.cuda.is_available():
        try:
            import faiss as _faiss
            _res = _faiss.StandardGpuResources()
            _cpu_idx = faiss_index._index
            faiss_index._index = _faiss.index_cpu_to_gpu(_res, 0, _cpu_idx)
            del _cpu_idx; import gc; gc.collect()
            _on_gpu = True
        except Exception as _e:
            print(f"[dense] GPU move skipped ({_e}); keeping index on host")
    print(f"[dense] faiss ntotal={faiss_index.ntotal:,} dim={faiss_index.dim} on_gpu={_on_gpu} ({time.time()-t0:.1f}s)")
    if torch.cuda.is_available():
        print(f"[mem] GPU {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"[mem] host RSS {psutil.Process().memory_info().rss/1e9:.2f} GB")
    # BGE-m3 runs on CUDA via FlagEmbedding; fp16 on GPU for speed.
    query_encoder = BGEQueryEncoder("BAAI/bge-m3", use_fp16=torch.cuda.is_available())
    print("[dense] BGE-m3 query encoder ready (downloads weights on first encode)")

# --- graph expander (optional) ----------------------------------------
graph_expander = None
if USE_GRAPH and KG.exists():
    t0 = time.time()
    graph_expander = GraphExpander.from_graph_and_meta(str(KG), str(META))
    print(f"[graph] expander loaded ({time.time()-t0:.1f}s)")
elif USE_GRAPH:
    print(f"[graph] kg.gpickle not found at {KG}; graph expansion disabled")

# Base retriever. The advanced layer (next section) reads cfg.top_bm25 /
# cfg.top_dense / cfg.rrf_k as the per-variant candidate sizes. final_top_k
# is applied by the advanced layer, not the base, but we set it for parity.
cfg = RetrievalConfig(
    use_dense=faiss_index is not None,
    use_reranker=USE_RERANK,
    top_bm25=BM25_TOPK,
    top_dense=DENSE_TOPK,
    rrf_k=RRF_K,
    fused_top=CANDIDATE_TOPK,   # used by the advanced layer as candidate pool
    expanded_top=CANDIDATE_TOPK,
    final_top_k=FINAL_TOP_K,
)
base_retriever = HybridRetriever(fts, faiss_index=faiss_index, graph_expander=graph_expander,
                                 query_encoder=query_encoder, config=cfg)
print("[retriever] base HybridRetriever ready")


[fts] rows=636,585  backend='fts5'  mode='bm25_ranked'  (52.3s)
[gpu] active device = cuda (Tesla T4)
[dense] faiss ntotal=636,585 dim=1024 on_gpu=True (58.4s)
[mem] GPU 0.00 GB / 15.6 GB
[mem] host RSS 2.19 GB
[dense] BGE-m3 query encoder ready (downloads weights on first encode)
[graph] expander loaded (44.5s)
[retriever] base HybridRetriever ready


## 4. Advanced retrieval layer

This is the integration of your teammate's advanced techniques from
[`hybridrag-advanced.ipynb`](notebooks/hybridrag-advanced.ipynb), re-targeted onto
the repo's [`HybridRetriever`](src/retrieval/retriever.py). It defines:

- **Text helpers** — Vietnamese normalisation, stopword list, `LEGAL_EXPANSIONS`
  domain synonyms (verbatim from the teammate's notebook).
- **Query-variant builders** — [`split_question_clauses`](src/retrieval/retriever.py),
  [`keyword_rewrite`](src/retrieval/retriever.py), [`expansion_query`](src/retrieval/retriever.py),
  [`build_query_variants`](src/retrieval/retriever.py) (weighted, deduped, capped).
- **Weighted RRF** — generalises the repo's unweighted [`rrf_fuse`](src/retrieval/rrf.py)
  with per-variant weights.
- **Article aggregation** — [`aggregate_article_contexts`](src/retrieval/retriever.py) +
  [`adaptive_article_topk`](src/retrieval/retriever.py) + [`make_relevant_lists_from_ctx`](src/retrieval/retriever.py).
- **Optional HyDE** — [`HyDEGenerator`](src/retrieval/retriever.py) (Qwen2.5-7B).
- **[`AdvancedHybridRetriever`](src/retrieval/retriever.py)** — orchestrates everything by calling the
  base retriever's legs once per variant, fusing with weighted RRF, running the base
  reranker, then (optionally) article-aggregating. Returns `List[Hit]` so downstream
  cells are unchanged.

> Design note: the advanced layer calls `base_retriever._lexical_leg` /
> `base_retriever._dense_leg` directly (one call per variant) rather than re-implementing
> FTS/FAISS. The base reranker (`base_retriever._rerank`) is reused once on the fused
> candidate pool — so the cross-encoder weights are loaded only once.


In [9]:
# =========================================================
# 4a. TEXT HELPERS + VIETNAMESE LEGAL EXPANSION
# (ported from hybridrag-advanced.ipynb cell 1)
# =========================================================
import re
from collections import OrderedDict, defaultdict

_word_re = re.compile(r'\w+', re.UNICODE)

STOPWORDS = {
    'và', 'hoặc', 'của', 'các', 'những', 'một', 'này', 'đó', 'thì', 'là', 'có', 'bị', 'được',
    'phải', 'cho', 'về', 'trong', 'ngoài', 'theo', 'nếu', 'khi', 'như', 'để', 'với', 'từ', 'ra',
    'sao', 'gì', 'nào', 'bao', 'nhiêu', 'trường', 'hợp', 'cần', 'muốn', 'hỏi', 'tôi', 'công', 'ty'
}

# Domain synonym rules for Vietnamese legal queries. When the question
# matches the pattern, the replacement terms are injected into the keyword
# variant to widen lexical recall (e.g. "thuế" -> thuế TNDN + GTGT + miễn giảm).
LEGAL_EXPANSIONS = [
    (r'\bưu đãi\b', 'hỗ trợ miễn giảm chính sách ưu đãi'),
    (r'\bhỗ trợ\b', 'chính sách hỗ trợ doanh nghiệp nhỏ và vừa'),
    (r'\bđấu thầu\b', 'lựa chọn nhà thầu ưu đãi trong lựa chọn nhà thầu'),
    (r'\bphạt|xử lý|vi phạm\b', 'xử phạt vi phạm hành chính biện pháp khắc phục hậu quả'),
    (r'\bgiữ\b.*\b(bằng|văn bằng|chứng chỉ|giấy tờ)\b', 'giữ bản chính giấy tờ tùy thân văn bằng chứng chỉ của người lao động'),
    (r'\bhợp đồng lao động\b', 'giao kết thực hiện chấm dứt hợp đồng lao động'),
    (r'\bthuế\b', 'thuế thu nhập doanh nghiệp thuế giá trị gia tăng miễn giảm thuế'),
    (r'\bbảo hiểm\b', 'bảo hiểm xã hội bảo hiểm y tế bảo hiểm thất nghiệp'),
    (r'\bhồ sơ\b', 'thành phần hồ sơ tài liệu chứng cứ đơn yêu cầu'),
    (r'\bthủ tục\b', 'trình tự thủ tục hồ sơ cơ quan có thẩm quyền'),
    (r'\bthời hạn\b', 'thời hạn thời hiệu số ngày làm việc'),
    (r'\bquyền tác giả|phần mềm|sao chép\b', 'quyền tác giả chương trình máy tính hành vi xâm phạm thiệt hại chứng cứ'),
]

def normalize_text(text):
    text = str(text).replace('\u200b', ' ').replace('\ufeff', ' ')
    return re.sub(r'\s+', ' ', text).strip()

def normalize_key(text):
    return normalize_text(text).lower()

def tokenize_lexical(text, keep_stopwords=False):
    toks = _word_re.findall(normalize_text(text).lower())
    toks = [t for t in toks if len(t) >= 2]
    if not keep_stopwords:
        toks = [t for t in toks if t not in STOPWORDS]
    return toks

print("[advanced] text helpers + %d legal expansion rules loaded" % len(LEGAL_EXPANSIONS))


[advanced] text helpers + 12 legal expansion rules loaded


In [10]:
# =========================================================
# 4b. QUERY VARIANTS
# (ported from hybridrag-advanced.ipynb cell 3)
# =========================================================

def split_question_clauses(question):
    """Split a verbose question into sub-clauses for per-clause retrieval."""
    q = normalize_text(question)
    pieces = re.split(r'[;?。]+|\s+(?:đồng thời|ngoài ra|bên cạnh đó|trong trường hợp|nếu|khi|và nếu)\s+',
                      q, flags=re.IGNORECASE)
    out = []
    for piece in pieces:
        piece = normalize_text(piece.strip(' ,.-:'))
        if 25 <= len(piece) <= 260:
            out.append(piece)
    if len(out) <= 1 and len(q) > 120:
        for piece in re.split(r',\s+', q):
            piece = normalize_text(piece.strip(' ,.-:'))
            if 25 <= len(piece) <= 220:
                out.append(piece)
    seen, deduped = set(), []
    for piece in out:
        key = normalize_key(piece)
        if key not in seen and key != normalize_key(q):
            seen.add(key)
            deduped.append(piece)
    return deduped[:4]

def keyword_rewrite(question):
    """Tokenise + inject legal-expansion synonyms; used as the keyword variant."""
    toks = tokenize_lexical(question, keep_stopwords=False)
    expansions = []
    q_lower = normalize_key(question)
    for pat, repl in LEGAL_EXPANSIONS:
        if re.search(pat, q_lower, flags=re.IGNORECASE):
            expansions.extend(tokenize_lexical(repl, keep_stopwords=False))
    merged = []
    for tok in toks + expansions:
        if tok not in merged:
            merged.append(tok)
    if not merged:
        return normalize_text(question)
    return ' '.join(merged[:48])

def expansion_query(question):
    """Original question + concatenated expansion phrases (dense-friendly)."""
    q_lower = normalize_key(question)
    hits = []
    for pat, repl in LEGAL_EXPANSIONS:
        if re.search(pat, q_lower, flags=re.IGNORECASE):
            hits.append(repl)
    if not hits:
        return ''
    return normalize_text(question + ' ' + ' '.join(hits[:4]))

def add_variant(variants, text, kind, weight, dense_only=False):
    text = normalize_text(text)
    if not text:
        return
    key = normalize_key(text)
    if key in {v['key'] for v in variants}:
        return
    variants.append({
        'text': text,
        'kind': kind,
        'weight': float(weight),
        'dense_only': bool(dense_only),
        'key': key,
    })

def build_query_variants(question, hyde_text=''):
    """Weighted, deduped query variants for one question.

    kinds/weights (mirrors hybridrag-advanced.ipynb):
      original 1.00 | keyword 0.92 | expansion 0.82 | clause 0.86 |
      clause_keyword 0.78 | hyde 0.62 (dense-only)
    """
    variants = []
    add_variant(variants, question, 'original', 1.00)
    add_variant(variants, keyword_rewrite(question), 'keyword', 0.92)
    exp_q = expansion_query(question)
    if exp_q:
        add_variant(variants, exp_q, 'expansion', 0.82)
    for clause in split_question_clauses(question):
        add_variant(variants, clause, 'clause', 0.86)
        add_variant(variants, keyword_rewrite(clause), 'clause_keyword', 0.78)
    if hyde_text:
        add_variant(variants, hyde_text, 'hyde', 0.62, dense_only=True)
    return variants[:MAX_QUERY_VARIANTS]

print("[advanced] query-variant builders loaded")


[advanced] query-variant builders loaded


In [11]:
# =========================================================
# 4c. WEIGHTED RRF + ARTICLE AGGREGATION
# (ported from hybridrag-advanced.ipynb cell 4)
# =========================================================
import math

def weighted_rrf_fuse(weighted_rankings, k=60, topk=96):
    """RRF with per-ranking weights. Generalises retrieval.rrf.rrf_fuse.

    Each entry is (ranking, weight, label). Score contribution of a hit at
    1-based rank r in a ranking with weight w is w / (k + r).
    """
    scores = defaultdict(float)
    payload = {}
    sources = defaultdict(list)
    for ranking, weight, label in weighted_rankings:
        for rank, item in enumerate(ranking, start=1):
            row_idx = int(item['row_idx'])
            scores[row_idx] += float(weight) / (k + rank)
            payload.setdefault(row_idx, item)
            sources[row_idx].append({'source': label, 'rank': rank, 'weight': float(weight)})
    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:topk]
    out = []
    for row_idx, score in fused:
        item = dict(payload[row_idx])
        item['rrf_score'] = float(score)
        item['source_hits'] = sources[row_idx][:8]
        out.append(item)
    return out

def article_key(c):
    law_id = str(c.get('law_id', '')).strip()
    ten = str(c.get('ten_van_ban', '')).strip()
    dieu = str(c.get('dieu_so', '')).strip()
    if law_id and ten and dieu:
        return f'{law_id}|{ten}|{dieu}'
    return ''

def adaptive_article_topk(question):
    """Widen the article budget for long / multi-clause questions."""
    topk = ARTICLE_CTX_TOPK
    toks = tokenize_lexical(question, keep_stopwords=True)
    q = normalize_key(question)
    multi_markers = len(re.findall(
        r'\b(và|đồng thời|ngoài ra|nếu|khi|hồ sơ|chứng cứ|xử lý|khắc phục|nghĩa vụ|trách nhiệm)\b', q))
    if len(toks) >= 45:
        topk += 4
    if multi_markers >= 3:
        topk += 4
    return min(topk, ARTICLE_CTX_MAX)

def aggregate_article_contexts(reranked, question):
    """Group chunks by article, keep the best chunk per article, score with
    support-count + RRF + early-rank bonuses (teammate's heuristic)."""
    groups = OrderedDict()
    for rank, c in enumerate(reranked, start=1):
        key = article_key(c)
        if not key:
            continue
        if key not in groups:
            groups[key] = {
                'article_key': key,
                'law_id': str(c.get('law_id', '')).strip(),
                'ten_van_ban': str(c.get('ten_van_ban', '')).strip(),
                'dieu_so': str(c.get('dieu_so', '')).strip(),
                'best_context': dict(c),
                'best_rerank_score': float(c.get('rerank_score', c.get('score', -1e9))),
                'best_rrf_score': float(c.get('rrf_score', 0.0)),
                'best_rank': rank,
                'support_count': 0,
                'support_rrf_sum': 0.0,
            }
        g = groups[key]
        g['support_count'] += 1
        g['support_rrf_sum'] += float(c.get('rrf_score', 0.0))
        best_score = float(c.get('rerank_score', c.get('score', -1e9)))
        if best_score > g['best_rerank_score']:
            g['best_context'] = dict(c)
            g['best_rerank_score'] = best_score
            g['best_rrf_score'] = float(c.get('rrf_score', 0.0))
            g['best_rank'] = rank

    article_contexts = []
    for g in groups.values():
        support_bonus = 0.06 * math.log1p(g['support_count'])
        rrf_bonus = 0.30 * g['support_rrf_sum']
        early_bonus = 0.03 / max(g['best_rank'], 1)
        g['article_score'] = float(g['best_rerank_score'] + support_bonus + rrf_bonus + early_bonus)
        c = dict(g['best_context'])
        c['article_key'] = g['article_key']
        c['article_score'] = g['article_score']
        c['support_count'] = g['support_count']
        c['support_rrf_sum'] = float(g['support_rrf_sum'])
        article_contexts.append(c)
    article_contexts.sort(key=lambda x: x['article_score'], reverse=True)
    return article_contexts[:adaptive_article_topk(question)]

print("[advanced] weighted RRF + article aggregation loaded")


[advanced] weighted RRF + article aggregation loaded


In [12]:
# =========================================================
# 4d. OPTIONAL HyDE (Qwen2.5-7B hypothetical document)
# (ported from hybridrag-advanced.ipynb cell 3, HyDEGenerator)
# =========================================================
class HyDEGenerator:
    """Generate a hypothetical legal passage to use as a dense-only query variant.

    Lazily loads Qwen2.5-7B-Instruct (fp16 on GPU). Only constructed when
    USE_HYDE is True; otherwise this cell defines but never instantiates it.
    """
    def __init__(self, model_name=HYDE_MODEL):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
            device_map='auto' if device == 'cuda' else None,
            trust_remote_code=True,
        )
        self.model.eval()
        print(f"[hyde] generator ready on {device} ({model_name})")

    def generate(self, question):
        import torch
        system = 'Bạn viết truy vấn pháp lý giả định để hỗ trợ tìm kiếm văn bản luật Việt Nam. Không bịa số điều, không nêu citation.'
        user = (
            'Câu hỏi: ' + str(question) + '\n\n'
            'Hãy viết một đoạn 2-4 câu bằng văn phong pháp lý, nêu các khái niệm và thuật ngữ có thể xuất hiện trong văn bản luật liên quan. '
            'Không kết luận, không nêu số điều/khoản cụ thể.'
        )
        if hasattr(self.tokenizer, 'apply_chat_template'):
            prompt = self.tokenizer.apply_chat_template([
                {'role': 'system', 'content': system},
                {'role': 'user', 'content': user},
            ], tokenize=False, add_generation_prompt=True)
        else:
            prompt = system + '\n\n' + user + '\n\nTruy vấn giả định:'
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=HYDE_MAX_NEW_TOKENS,
                temperature=0.2,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.02,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        gen_ids = out[0][inputs['input_ids'].shape[1]:]
        text = self.tokenizer.decode(gen_ids, skip_special_tokens=True)
        return normalize_text(re.sub(r'(?is)^(assistant|truy vấn giả định)\s*:?', '', text))

print("[advanced] HyDE generator class defined (USE_HYDE=%s)" % USE_HYDE)


[advanced] HyDE generator class defined (USE_HYDE=True)


In [13]:
# =========================================================
# 4e. AdvancedHybridRetriever — the integrated orchestrator
# =========================================================
from retrieval.retriever import Hit
from retrieval.debug import (
    RetrievalTrace, StageSnapshot, snapshot_items,
    format_trace as _fmt_trace,
)

class AdvancedHybridRetriever:
    """Multi-query + weighted-RRF + article-aggregation layer over HybridRetriever.

    Pipeline per query:
      1. build query variants (original/keyword/expansion/clause[/hyde])
      2. for each variant: run the base retriever's lexical leg (+ dense leg
         for non dense-only variants) — reusing base._lexical_leg / _dense_leg
      3. weighted-RRF fuse all variant rankings -> candidate pool (CANDIDATE_TOPK)
      4. optional graph expansion (base.graph_expander) on the fused pool
      5. fetch metadata + text via base._fetch_meta / base._fetch_text
      6. optional cross-encoder rerank (base._rerank) -> RERANK_TOPK
      7. optional article aggregation -> one best chunk per article
      8. return List[Hit] (FINAL_TOP_K), so make_relevant_lists works unchanged

    If USE_MULTIQUERY is False this collapses to the base retrieve() path
    (single original query) — i.e. identical behaviour to retrieval_colab.ipynb.
    """

    def __init__(self, base, use_multiquery=True, use_article_agg=True):
        self.base = base
        self.use_multiquery = use_multiquery
        self.use_article_agg = use_article_agg
        self._hyde = None  # lazily built
        self.last_trace = None  # per-stage debug trace (set when explain=True)

    # -- HyDE ---------------------------------------------------------- #
    def _hyde_text(self, question):
        if not USE_HYDE:
            return ''
        if self._hyde is None:
            self._hyde = HyDEGenerator()
        try:
            return self._hyde.generate(question)
        except Exception as e:
            print('[hyde] failed, skipping:', repr(e))
            return ''

    # -- per-variant legs --------------------------------------------- #
    def _variant_legs(self, variant):
        """Run lexical (+ dense unless dense_only) for one variant; return
        (lexical_hits, dense_hits) lists of dicts keyed by row_idx."""
        cfg = self.base.config
        lex = self.base._lexical_leg(variant['text'], cfg.top_bm25)
        dense = []
        if cfg.use_dense and self.base.faiss_index is not None and not variant.get('dense_only'):
            dense = self.base._dense_leg(variant['text'], cfg.top_dense)
        return lex, dense

    @property
    def last_trace_formatted(self):
        return _fmt_trace(self.last_trace) if self.last_trace is not None else None

    # -- main entry ---------------------------------------------------- #
    def retrieve(self, query, fetch_text=False, explain=False, print_trace=False):
        cfg = self.base.config

        # ---- debug trace setup (zero overhead when explain=False) ----------
        trace = None
        if explain:
            import time as _time
            _t0 = _time.perf_counter()
            trace = RetrievalTrace(query=query, config={
                'use_dense': cfg.use_dense, 'use_reranker': cfg.use_reranker,
                'graph': self.base.graph_expander is not None,
                'use_multiquery': self.use_multiquery,
                'use_article_agg': self.use_article_agg,
                'top_bm25': cfg.top_bm25, 'top_dense': cfg.top_dense,
                'fused_top': cfg.fused_top, 'final_top_k': cfg.final_top_k,
                'rrf_k': cfg.rrf_k,
            })

        # ---- 0. single-query fast path (collapses to base behaviour) ----
        if not self.use_multiquery:
            hits = self.base.retrieve(query, fetch_text=fetch_text)
            if trace is not None:
                trace.add(StageSnapshot(
                    name='base_retrieve', count=len(hits),
                    top_items=[{'row_idx': h.row_idx, 'score': h.score,
                                'source': h.source, 'law_id': h.law_id,
                                'ten_van_ban': h.ten_van_ban, 'dieu_so': h.dieu_so}
                               for h in hits[:8]],
                    skip='use_multiquery=False -> base.retrieve()',
                ))
                trace.total_elapsed_ms = (_time.perf_counter() - _t0) * 1000.0
                self.last_trace = trace
                if print_trace:
                    print(_fmt_trace(trace))
            return hits

        # ---- 1. query variants + HyDE --------------------------------
        if trace is not None:
            t0 = _time.perf_counter()
        hyde_text = self._hyde_text(query)
        variants = build_query_variants(query, hyde_text)
        print(f"[advanced] {len(variants)} variants: " +
              ', '.join(f"{v['kind']}({v['weight']})" for v in variants))
        if trace is not None:
            trace.add(StageSnapshot(
                name='variants', count=len(variants),
                elapsed_ms=(_time.perf_counter() - t0) * 1000.0,
                top_items=[{'row_idx': i, 'score': v['weight'], 'source': v['kind'],
                            'text': (v['text'][:90] + ('...' if len(v['text']) > 90 else ''))}
                           for i, v in enumerate(variants[:8])],
                diagnostics={'hyde': bool(hyde_text),
                             'dense_only_kinds': [v['kind'] for v in variants if v.get('dense_only')]},
            ))

        # ---- 2. per-variant legs -> weighted rankings ----------------
        if trace is not None:
            t0 = _time.perf_counter()
        weighted_rankings = []
        per_variant_counts = []
        for v in variants:
            lex, dense = self._variant_legs(v)
            per_variant_counts.append({'kind': v['kind'], 'lexical': len(lex), 'dense': len(dense)})
            if lex:
                weighted_rankings.append((lex, v['weight'], f"lexical:{v['kind']}"))
            if dense:
                weighted_rankings.append((dense, v['weight'], f"dense:{v['kind']}"))
        if trace is not None:
            trace.add(StageSnapshot(
                name='legs', count=len(weighted_rankings),
                elapsed_ms=(_time.perf_counter() - t0) * 1000.0,
                diagnostics={'per_variant': per_variant_counts,
                             'lexical_rankings': sum(1 for r in weighted_rankings if r[2].startswith('lexical')),
                             'dense_rankings': sum(1 for r in weighted_rankings if r[2].startswith('dense'))},
            ))

        # ---- 3. weighted RRF fuse ------------------------------------
        if trace is not None:
            t0 = _time.perf_counter()
        fused = weighted_rrf_fuse(weighted_rankings, k=cfg.rrf_k, topk=cfg.fused_top)
        if trace is not None:
            trace.add(StageSnapshot(
                name='weighted_rrf', count=len(fused),
                elapsed_ms=(_time.perf_counter() - t0) * 1000.0,
                top_items=snapshot_items(
                    [{**f, 'score': f.get('rrf_score', 0.0)} for f in fused], 8,
                    score_key='rrf_score',
                    extra_keys=('law_id', 'ten_van_ban', 'dieu_so')),
                diagnostics={'k': cfg.rrf_k, 'topk': cfg.fused_top,
                             'rankings_in': len(weighted_rankings)},
            ))
        if not fused:
            if trace is not None:
                trace.total_elapsed_ms = (_time.perf_counter() - _t0) * 1000.0
                self.last_trace = trace
            return []

        # ---- 4. graph expansion (optional, reuse base) ---------------
        if trace is not None:
            t0 = _time.perf_counter()
        if self.base.graph_expander is not None:
            candidates = [(r['row_idx'], r['rrf_score']) for r in fused]
            expanded = self.base.graph_expander.expand(candidates, top_n=cfg.expanded_top)
            ordered = [{**e, 'score': e.get('score', 0.0)} for e in expanded]
        else:
            ordered = [{'row_idx': r['row_idx'], 'score': r['rrf_score'],
                        'source': 'fused', 'rrf_score': r['rrf_score']} for r in fused]
        ordered.sort(key=lambda e: (-e['score'], e['row_idx']))
        if trace is not None:
            src_counts = {}
            for e in ordered:
                sk = str(e.get('source', '?'))
                src_counts[sk] = src_counts.get(sk, 0) + 1
            trace.add(StageSnapshot(
                name='graph', count=len(ordered),
                elapsed_ms=(_time.perf_counter() - t0) * 1000.0,
                top_items=snapshot_items(ordered, 8, score_key='score',
                    extra_keys=('source', 'law_id', 'ten_van_ban', 'dieu_so')),
                skip='no graph_expander' if self.base.graph_expander is None else None,
                diagnostics={'expanded_top': cfg.expanded_top, 'source_counts': src_counts},
            ))

        # ---- 5. fetch metadata + optional text -----------------------
        if trace is not None:
            t0 = _time.perf_counter()
        row_idxs = [e['row_idx'] for e in ordered]
        meta_by_row = self.base._fetch_meta(row_idxs)
        text_by_row = self.base._fetch_text(row_idxs) if fetch_text else {}
        for e in ordered:
            e['rrf_score'] = e.get('rrf_score', e.get('score', 0.0))
        if trace is not None:
            missing = [ri for ri in row_idxs if ri not in meta_by_row]
            ordered_meta = [{**e, **meta_by_row.get(e['row_idx'], {})} for e in ordered]
            trace.add(StageSnapshot(
                name='fetch', count=len(meta_by_row),
                elapsed_ms=(_time.perf_counter() - t0) * 1000.0,
                top_items=snapshot_items(ordered_meta, 8, score_key='score',
                    extra_keys=('source', 'law_id', 'ten_van_ban', 'dieu_so', 'chunk_id', 'doc_uid')),
                diagnostics={'requested': len(row_idxs), 'resolved': len(meta_by_row),
                             'missing_rows': missing, 'text_fetched': bool(fetch_text)},
            ))

        # ---- 6. rerank (optional, reuse base) ------------------------
        if trace is not None:
            t0 = _time.perf_counter()
        pre = len(ordered)
        if cfg.use_reranker and ordered:
            ordered = self.base._rerank(query, ordered, meta_by_row, text_by_row)
        ordered = ordered[:RERANK_TOPK]
        if trace is not None:
            skip = None if (cfg.use_reranker and pre) else (
                'use_reranker=False' if not cfg.use_reranker else 'no candidates to rerank')
            trace.add(StageSnapshot(
                name='rerank', count=len(ordered),
                elapsed_ms=(_time.perf_counter() - t0) * 1000.0,
                top_items=snapshot_items(ordered, 8, score_key='score',
                    extra_keys=('rerank_score', 'source', 'law_id', 'ten_van_ban', 'dieu_so')),
                skip=skip,
                diagnostics={'candidates_in': pre, 'rerank_topk': RERANK_TOPK,
                             'reranker_unavailable': self.base._reranker_unavailable},
            ))

        # ---- 7. article aggregation (optional) -----------------------
        if trace is not None:
            t0 = _time.perf_counter()
        if self.use_article_agg:
            for e in ordered:
                m = meta_by_row.get(int(e['row_idx']), {})
                e.update({k: m.get(k, e.get(k, '')) for k in
                          ('law_id', 'ten_van_ban', 'dieu_so', 'chunk_id', 'doc_uid')})
                if fetch_text:
                    e['chunk_text'] = text_by_row.get(int(e['row_idx']), e.get('chunk_text', ''))
            agg = aggregate_article_contexts(ordered, query)
            ordered = agg
        if trace is not None:
            items = []
            for e in ordered[:8]:
                items.append({'row_idx': int(e['row_idx']),
                              'score': float(e.get('article_score', e.get('score', 0.0))),
                              'source': str(e.get('source', '')),
                              'law_id': e.get('law_id', ''),
                              'ten_van_ban': e.get('ten_van_ban', ''),
                              'dieu_so': e.get('dieu_so', ''),
                              'support_count': e.get('support_count')})
            trace.add(StageSnapshot(
                name='article_agg', count=len(ordered),
                elapsed_ms=(_time.perf_counter() - t0) * 1000.0,
                top_items=items,
                skip='use_article_agg=False' if not self.use_article_agg else None,
                diagnostics={'adaptive_topk': adaptive_article_topk(query)},
            ))

        # ---- 8. build final List[Hit] --------------------------------
        hits = []
        for e in ordered[:cfg.final_top_k]:
            row_idx = int(e['row_idx'])
            m = meta_by_row.get(row_idx, {})
            hits.append(Hit(
                row_idx=row_idx,
                score=float(e.get('score', e.get('article_score', 0.0))),
                source=str(e.get('source', 'fused')),
                law_id=str(e.get('law_id', m.get('law_id', ''))),
                ten_van_ban=str(e.get('ten_van_ban', m.get('ten_van_ban', ''))),
                dieu_so=str(e.get('dieu_so', m.get('dieu_so', ''))),
                chunk_id=str(e.get('chunk_id', m.get('chunk_id', ''))),
                doc_uid=str(e.get('doc_uid', m.get('doc_uid', ''))),
                chunk_text=text_by_row.get(row_idx, '') if fetch_text else '',
            ))

        if trace is not None:
            trace.add(StageSnapshot(
                name='final', count=len(hits),
                top_items=[{'row_idx': h.row_idx, 'score': h.score, 'source': h.source,
                            'law_id': h.law_id, 'ten_van_ban': h.ten_van_ban,
                            'dieu_so': h.dieu_so, 'chunk_id': h.chunk_id}
                           for h in hits[:8]],
                diagnostics={'final_top_k': cfg.final_top_k},
            ))
            docs, articles = make_relevant_lists(hits)
            trace.add(StageSnapshot(
                name='output', count=len(hits),
                diagnostics={'relevant_docs': docs, 'relevant_articles': articles},
            ))
            trace.total_elapsed_ms = (_time.perf_counter() - _t0) * 1000.0
            self.last_trace = trace
            if print_trace:
                print(_fmt_trace(trace))

        return hits

advanced_retriever = AdvancedHybridRetriever(
    base_retriever,
    use_multiquery=USE_MULTIQUERY,
    use_article_agg=USE_ARTICLE_AGG,
)
print("[retriever] AdvancedHybridRetriever ready "
      f"(multiquery={USE_MULTIQUERY}, article_agg={USE_ARTICLE_AGG}, hyde={USE_HYDE})")


[retriever] AdvancedHybridRetriever ready (multiquery=True, article_agg=True, hyde=True)


## 5. Run a query

Edit `QUERY` and run. Returns the final top-K hits with metadata + chunk text,
plus the collapsed `relevant_docs` / `relevant_articles`. With
`USE_MULTIQUERY=True` you'll also see the generated variants printed above the hits.


In [ ]:
# ──────────────────────────  EDIT YOUR QUERY HERE  ──────────────────────────
QUERY = "đối thủ sao chép trái phép phần mềm để cho thuê thu lợi và làm mất khách hàng; cần xác định hành vi xâm phạm quyền tác giả ở điểm nào, cách tính tổn thất về cơ hội kinh doanh ra sao, và phải chuẩn bị tài liệu / chứng cứ gì khi gửi đơn yêu cầu xử lý?"
# ────────────────────────────────────────────────────────────────────────────

TEXT_PREVIEW = 800   # chars of chunk_text to show per hit (0 = full text)

t0 = time.time()
hits = advanced_retriever.retrieve(QUERY, fetch_text=True)
print(f"\nRetrieved {len(hits)} hits in {time.time()-t0:.2f}s\n")

for i, h in enumerate(hits, 1):
    print(f"#{i}  row_idx={h.row_idx}  score={h.score:+.4f}  source={h.source}")
    print(f"    law_id    : {h.law_id}")
    print(f"    ten_van_ban: {h.ten_van_ban}")
    print(f"    dieu_so   : {h.dieu_so}")
    txt = h.chunk_text or ""
    if TEXT_PREVIEW and len(txt) > TEXT_PREVIEW:
        txt = txt[:TEXT_PREVIEW] + " …[truncated]"
    print(f"    chunk_text:\n{txt}")
    print("-" * 100)

docs, articles = make_relevant_lists(hits)
print("\nrelevant_docs:", docs)
print("relevant_articles:", articles)


/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

Constant_7_attr__value:   0%|          | 0.00/65.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

onnx/model.onnx:   0%|          | 0.00/725k [00:00<?, ?B/s]

onnx/model.onnx_data:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

## 5b. 🔬 Explainability — per-stage debug trace

When retrieval quality is off, you want to see *what each stage produced*, not just the final hits.
The advanced retriever now supports `explain=True`, which records a `RetrievalTrace` with one
`StageSnapshot` per stage of the multi-query pipeline and stores it on `advanced_retriever.last_trace`.

Stages traced (in order):

| Stage | What it records |
|---|---|
| `variants` | generated query variants (kind, weight, text preview) + HyDE on/off |
| `legs` | per-variant lexical/dense hit counts + how many rankings feed RRF |
| `weighted_rrf` | fused candidate pool (top-N with rrf_score + legal metadata) + k/topk |
| `graph` | post-expansion candidates + `source_counts` (candidate/doc_expand/concept_expand) |
| `fetch` | requested vs resolved `row_idx`, `missing_rows`, text_fetched |
| `rerank` | post cross-encoder order (rerank_score) + unavailable/skip reason |
| `article_agg` | one best chunk per article (article_score, support_count) + adaptive top-k |
| `final` / `output` | final `List[Hit]` + collapsed `relevant_docs` / `relevant_articles` |

Each stage also carries wall-clock `ms` and a `skip` reason when a stage is turned off or degrades,
so you can tell the difference between *a stage was disabled* and *a stage ran but found nothing*.

> Zero overhead when `explain=False` (the normal `retrieve()` path is unchanged). The helpers live in
> [`src/retrieval/debug.py`](src/retrieval/debug.py) and are reused by the base `HybridRetriever` too.


In [ ]:
# Run the SAME query as Section 5, but with the per-stage trace printed.
# explain=True records advanced_retriever.last_trace; print_trace=True also prints it.
t0 = time.time()
expl_hits = advanced_retriever.retrieve(QUERY, fetch_text=True, explain=True, print_trace=True)
print(f"\nExplained retrieve: {len(expl_hits)} hits in {time.time()-t0:.2f}s")

# The structured trace is also available for programmatic inspection / diffing:
trace = advanced_retriever.last_trace
if trace is not None:
    print("\nstage -> count  elapsed_ms  skip")
    for s in trace.stages:
        print(f"  {s.name:12s} {s.count:5d}  {str(s.elapsed_ms):>8s}  {s.skip or ''}")
    # e.g. inspect which articles survived article aggregation:
    out = trace.stage('output')
    if out is not None:
        print("\nrelevant_articles:", out.diagnostics.get('relevant_articles'))


## 6. (Optional) Load a dev-set example question

If you uploaded `dev_set/questions.json`, pick one of the 20 questions by index.


In [ ]:
import json

if DEV.exists() and (DEV / "questions.json").exists():
    QUESTIONS = json.loads((DEV / "questions.json").read_text(encoding="utf-8"))
    for q in QUESTIONS:
        print(f"{q['id']:>2}. {q['question']}")
else:
    print(f"dev_set not found at {DEV}. Upload dev_set/questions.json to use the picker.")
    QUESTIONS = []


In [ ]:
PICK = 1   # 1..20

if QUESTIONS:
    QUERY = QUESTIONS[PICK - 1]["question"]
    print("QUERY set to:\n", QUERY)
else:
    print("No dev questions loaded; set QUERY manually in the cell above.")


## 7. (Optional) Batch-run the whole dev set + F2 evaluation

Runs the advanced retrieval for all 20 dev questions and scores with the repo's
[`dev_set/eval.py`](src/../dev_set/eval.py) F2 macro metric. Requires
`dev_set/questions.json` + `dev_set/ground_truth.json`.

> Compare this F2 against the baseline notebook's F2 to measure the gain from
> the multi-query + article-aggregation layer.


In [ ]:
import sys as _sys, json as _json
from types import SimpleNamespace

DEV_LOCAL = Path(REPO_DIR) / "dev_set"
# Prefer Drive copy if present (in case you edited it), else the cloned repo copy.
QPATH  = (DEV / "questions.json") if (DEV / "questions.json").exists() else (DEV_LOCAL / "questions.json")
GTPATH = (DEV / "ground_truth.json") if (DEV / "ground_truth.json").exists() else (DEV_LOCAL / "ground_truth.json")
print("questions:", QPATH, "| ground_truth:", GTPATH)

questions = _json.loads(QPATH.read_text(encoding="utf-8"))
records, t0 = [], time.time()
for q in questions:
    hits = advanced_retriever.retrieve(q["question"], fetch_text=False)
    docs, articles = make_relevant_lists(hits)
    records.append({"id": q["id"], "question": q["question"], "answer": "",
                    "relevant_docs": docs, "relevant_articles": articles})
print(f"\nRan {len(records)} questions in {time.time()-t0:.1f}s")

OUT = Path("/content/results_colab_advanced.json")
OUT.write_text(_json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")

# Score with the repo's evaluator.
_sys.path.insert(0, str(DEV_LOCAL.parent))  # so `import dev_set.eval` works
from dev_set.eval import f2_macro
gt = _json.loads(GTPATH.read_text(encoding="utf-8"))
print(f"\nF2 macro (advanced) = {f2_macro(records, gt):.4f}")
print("results written to", OUT)


## 8. Cleanup


In [ ]:
fts.close()
print("FTS index closed. Done.")
